# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the five example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

volatility_flag_success completed
spread_frequency_mismatch_block blocked
arima_forecast_block blocked
prompt_injection_unknown_capability blocked


vix_full_sample_lookahead completed


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

84 run records, 20 counted by the page, 892 events


total              84.000000
completed          31.000000
blocked            35.000000
failed              0.000000
rejected            3.000000
completed_share     0.369048
blocked_share       0.416667
failed_share        0.000000
rejected_share      0.035714
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

In [4]:
from quantlab.monitoring import (
    grounded_narratives,
    planner_refusals,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **planner_refusals(records),
    }
)

approved              28.000000
rejected               3.000000
approval_rate          0.903226
edited                 3.000000
edit_rate              0.107143
approval_median_s      0.041000
approval_p95_s       114.975350
replays                4.000000
matching               4.000000
replay_match_rate      1.000000
narratives            31.000000
grounded              31.000000
grounded_rate          1.000000
refused               23.000000
refused_share          0.273810
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,128
1,P01,PLANNER_REFUSED,BLOCK,23
2,V09,LOOK_AHEAD,APPROVE,21
3,V08,FREQUENCY_MISMATCH,BLOCK,13
4,V03,INVALID_PARAM,BLOCK,8
5,V02,UNKNOWN_CAPABILITY,BLOCK,5
6,E04,ALIGNMENT_LOSS,WARN,4
7,R01,REQUEST_TOO_LONG,BLOCK,1
8,V01,INVALID_PLAN,BLOCK,0
9,V04,TYPE_MISMATCH,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,35
1,flag_regime,C07,27
2,resample,C02,23
3,compute_returns,C03,16
4,rolling_volatility,C06,16
5,forward_return,C08,16
6,compare_groups,C09,16
7,hac_mean_difference,C10,16
8,spread,C05,4
9,difference,C04,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,18,3291.944444,4976.3,271.777778,801.80,0.000465,0.000872,0.008372
1,narrator,18,552.833333,1578.0,70.222222,184.10,0.000090,0.000231,0.001627
2,total,18,3844.777778,5958.0,342.000000,958.65,0.000555,0.001070,0.009999


planner_calls             18.000000
cache_hits                 4.000000
hit_rate                   0.222222
saved_input_tokens     17504.000000
saved_output_tokens     2985.000000
saved_cost                 0.003243
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,83,897.015663,7539.860,True
2,Validator,83,0.124819,0.328,True
3,Executor,31,120.972903,190.295,True
4,Narrator,31,859.545161,3061.700,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-18T20:13:32.358Z,20261018-201332-9829d7,blocked,Validator,V02,The plan has blocking issues: V02:s2
1,2026-10-18T20:13:32.331Z,20261018-201332-b596c4,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-18T20:13:32.301Z,20261018-201332-4cd171,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-18T19:56:19.146Z,20261018-195619-6692e7,blocked,Planner,P01,The plan has blocking issues: P01:plan
4,2026-10-18T19:35:41.168Z,20261018-193541-493753,blocked,Planner,P01,The plan has blocking issues: P01:plan
